# LoRA Fine-Tuning Evaluation Analysis

This notebook provides interactive analysis of LoRA evaluation results.
Load and visualize the evaluation metrics and predictions generated by main.py

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
OUTPUT_DIR = '../evaluation_results'

## 1. Load Evaluation Metrics

In [ ]:
# Load metrics.json
with open(f'{OUTPUT_DIR}/metrics.json', 'r') as f:
    metrics = json.load(f)

# Display evaluation metadata
metadata = metrics['evaluation_metadata']
print('Evaluation Metadata:')
print(f"Timestamp: {metadata['timestamp']}")
print(f"Baseline Model: {metadata['baseline_model']}")
print(f"LoRA Config: r={metadata['lora_config']['r']}, alpha={metadata['lora_config']['lora_alpha']}")

## 2. Compare Baseline vs LoRA Accuracies

In [ ]:
# Create comparison dataframe
comparison_data = []
for dataset, results in metrics['results'].items():
    comparison_data.append({
        'Dataset': dataset.upper(),
        'Baseline Accuracy': results['baseline']['accuracy'],
        'LoRA Accuracy': results['lora']['accuracy'],
        'Improvement': results['comparison']['improvement_percentage']
    })

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

## 3. Summary Statistics

In [ ]:
# Calculate summary statistics
baseline_avg = np.mean([results['baseline']['accuracy'] for results in metrics['results'].values()])
lora_avg = np.mean([results['lora']['accuracy'] for results in metrics['results'].values()])
improvement_avg = np.mean([results['comparison']['improvement_percentage'] for results in metrics['results'].values()])

print(f'Baseline Average Accuracy: {baseline_avg:.2%}')
print(f'LoRA Average Accuracy: {lora_avg:.2%}')
print(f'Average Improvement: +{improvement_avg:.2f}%')

## 4. Load Predictions

In [ ]:
# Load predictions.json
with open(f'{OUTPUT_DIR}/predictions.json', 'r') as f:
    predictions = json.load(f)

# Analyze predictions for each dataset
for dataset_name, dataset_predictions in predictions.items():
    baseline_correct = sum(1 for p in dataset_predictions if p['baseline_correct'])
    lora_correct = sum(1 for p in dataset_predictions if p['lora_correct'])
    improved = sum(1 for p in dataset_predictions if p['lora_correct'] and not p['baseline_correct'])
    degraded = sum(1 for p in dataset_predictions if not p['lora_correct'] and p['baseline_correct'])
    
    print(f'{dataset_name.upper()}:')
    print(f'  Baseline Correct: {baseline_correct}')
    print(f'  LoRA Correct: {lora_correct}')
    print(f'  Improved: {improved}, Degraded: {degraded}')
    print()